In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import os

In [4]:
import cv2 
import os 

In [5]:
import matplotlib.pyplot as plt


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
import numpy as np
import os

# Load the InceptionV3 model
net = InceptionV3(include_top=True, weights='imagenet')
net.summary()

# Load images from directory using ImageDataGenerator
image_directory = r'C:\SumMe\Augmentation\AugA'
input_size = net.input_shape[1:3]  # Get the input size (height, width)

datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
# Create a data generator for the images
augimds = datagen.flow_from_directory(image_directory,
                                      target_size=input_size,
                                      batch_size=32,  # Adjust the batch size if needed
                                      class_mode=None,
                                      shuffle=False) 

# Create a model that outputs the activations from the specified layer
layer_name = 'predictions'  # Example layer name in InceptionV3 (change as needed)
intermediate_layer_model = tf.keras.Model(inputs=net.input,
                                          outputs=net.get_layer(layer_name).output)

# Generate activations
features_train = intermediate_layer_model.predict(augimds)
# Reshape the features to ensure they are in 'rows' format (2D)
features_train = features_train.reshape(features_train.shape[0], -1)
np.save(os.path.join(image_directory, 'inceptionv3_features_train.npy'), features_train)

print(f"Features shape: {features_train.shape}")

# Print some sample features
print("Sample features (first 5 rows):")
print(features_train[:5])

Model: "inception_v3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 299, 299, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 149, 149, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 149, 149, 32  96         ['conv2d[0][0]']                 
 alization)                     )                                                      

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import os


# Load the ResNet50 model
net = ResNet50(include_top=True, weights='imagenet')
net.summary()

# Load images from directory using ImageDataGenerator
image_directory = r'C:\SumMe\Augmentation\AugA'
input_size = net.input_shape[1:3]  # Get the input size (height, width)

datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
# Create a data generator for the images
augimds = datagen.flow_from_directory(image_directory,
                                      target_size=input_size,
                                      batch_size=32,  # Adjust the batch size if needed
                                      class_mode=None,
                                      shuffle=False) 

# Generate activations
features_train = intermediate_layer_model.predict(augimds)
# Create a model that outputs the activations from the specified layer
layer_name = 'predictions'  # Example layer name in ResNet50
intermediate_layer_model = tf.keras.Model(inputs=net.input,
                                          outputs=net.get_layer(layer_name).output)

# Generate activations
features_train = intermediate_layer_model.predict(augimds)
# Reshape the features to ensure they are in 'rows' format (2D)
features_train = features_train.reshape(features_train.shape[0], -1)
np.save(os.path.join(image_directory, 'resnet50_features_train.npy'), features_train)


print(f"Features shape: {features_train.shape}")

# Print some sample features
print("Sample features (first 5 rows):")
print(features_train[:5])

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['input_2[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 112, 112, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

In [9]:
import numpy as np
import os

# Define the directory where your features are saved
image_directory = r'C:\SumMe\Augmentation\AugA'

# Load the features from the saved .npy files
features_resnet = np.load(os.path.join(image_directory, 'resnet50_features_train.npy'))
features_inceptionv3 = np.load(os.path.join(image_directory, 'inceptionv3_features_train.npy'))

# Ensure both feature arrays have the same number of samples
assert features_resnet.shape[0] == features_inceptionv3.shape[0], "Number of samples must be the same in both feature arrays."

# Concatenate the features along the feature dimension (axis=1)
features_combined = np.concatenate((features_resnet, features_inceptionv3), axis=1)

# Save the combined features to a new .npy file
np.save(os.path.join(image_directory, 'combined_features_train.npy'), features_combined)

print(f"Combined features shape: {features_combined.shape}")

# Print some sample combined features
print("Sample combined features (first 5 rows):")
print(features_combined[:5])

Combined features shape: (4494, 2000)
Sample combined features (first 5 rows):
[[3.6533274e-06 7.0964736e-05 2.9688117e-06 ... 1.6794354e-05
  1.3399706e-04 7.5715383e-05]
 [4.2937913e-06 2.0422563e-05 6.8897890e-07 ... 4.4327302e-05
  3.7317578e-04 4.7465428e-04]
 [1.3961168e-06 3.4230236e-06 5.0522622e-06 ... 8.9841502e-05
  9.8107744e-04 2.9860399e-04]
 [8.6586880e-09 2.4686406e-08 4.7705520e-08 ... 4.2516890e-06
  5.7171387e-06 1.5436246e-05]
 [3.1681699e-07 6.0099291e-07 7.4092998e-07 ... 1.9576750e-05
  3.5495497e-05 4.9262526e-05]]


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [11]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

# Define the directory where your features are saved
image_directory = r'C:\SumMe\Augmentation\AugA'

# Load the features from the saved .npy files
features_combined = np.load(os.path.join(image_directory, 'combined_features_train.npy'))

# Define the number of timesteps (sequence length) and features
timesteps = 1  # Assuming each sample is treated as a sequence of length 1 for this example
input_dim = features_combined.shape[1]  # Should be 2000

# Prepare data for LSTM
# For this example, we'll use the features directly as sequences of length 1
X = features_combined
y = np.random.randint(0, 100, size=(features_combined.shape[0],))  # Example target variable, replace with your actual labels

# Reshape input to be [samples, timesteps, features]
X = X.reshape((X.shape[0], timesteps, input_dim))

# Define the LSTM model
inputs = Input(shape=(timesteps, input_dim))

# Add LSTM layers
x = LSTM(64, return_sequences=True)(inputs)
x = Dropout(0.5)(x)

x = LSTM(128, return_sequences=True)(x)
x = Dropout(0.5)(x)

x = LSTM(256, return_sequences=True)(x)
x = Dropout(0.5)(x)

x = LSTM(256, return_sequences=True)(x)
x = Dropout(0.5)(x)

x = LSTM(128, return_sequences=True)(x)
x = Dropout(0.5)(x)

x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

x = LSTM(32)(x)
x = Dropout(0.5)(x)

# Add a Dense layer for output
outputs = Dense(100, activation='softmax')(x)  # Adjust the number of units based on your specific task

# Create the model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Train the model
# For this example, we use dummy targets. Replace y with your actual labels.
model.fit(X, y, epochs=20, batch_size=32, validation_split=0.2)

# Extract features from the last LSTM layer
feature_extractor = Model(inputs=model.input, outputs=model.layers[-3].output)  # Layer before the final Dense layer

# Obtain features from the last LSTM layer
features_last_lstm = feature_extractor.predict(X)

# Save the extracted features
np.save(os.path.join(image_directory, 'features_last_lstm.npy'), features_last_lstm)

# Print the feature shape
print(f"Shape of features extracted from the last LSTM layer: {features_last_lstm.shape}")

# Print some sample features
print("Sample features from the last LSTM layer (first 5 samples):")
print(features_last_lstm[:5])

Epoch 1/20
113/113 [==============================] - 24s 61ms/step - loss: 4.6055 - accuracy: 0.0072 - val_loss: 4.6052 - val_accuracy: 0.0111
Epoch 2/20
113/113 [==============================] - 4s 39ms/step - loss: 4.6038 - accuracy: 0.0117 - val_loss: 4.6050 - val_accuracy: 0.0111
Epoch 3/20
113/113 [==============================] - 4s 40ms/step - loss: 4.6025 - accuracy: 0.0097 - val_loss: 4.6055 - val_accuracy: 0.0111
Epoch 4/20
113/113 [==============================] - 5s 41ms/step - loss: 4.6002 - accuracy: 0.0128 - val_loss: 4.6085 - val_accuracy: 0.0089
Epoch 5/20
113/113 [==============================] - 5s 40ms/step - loss: 4.5999 - accuracy: 0.0089 - val_loss: 4.6069 - val_accuracy: 0.0089
Epoch 6/20
113/113 [==============================] - 4s 38ms/step - loss: 4.5988 - accuracy: 0.0111 - val_loss: 4.6082 - val_accuracy: 0.0089
Epoch 7/20
113/113 [==============================] - 4s 37ms/step - loss: 4.5980 - accuracy: 0.0128 - val_loss: 4.6077 - val_accuracy: 0.011

In [13]:
import numpy as np
from sklearn.cluster import KMeans

# Load data (assuming the data is saved in a numpy array .npy file)
# Replace the path with the actual location of your data file
F = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")  # Adjust this path if necessary

# Set parameters
N = 675
np.random.seed(2)  # Setting the random seed for reproducibility

# Perform k-means clustering
kmeans = KMeans(n_clusters=N, init='k-means++', random_state=2)
kmeans.fit(F)

# Get cluster assignments (C_IDX) and cluster centroids (CC)
C_IDX = kmeans.labels_
CC = kmeans.cluster_centers_

# Initialize list to store indices of data points in each cluster
C = {i: np.where(C_IDX == i)[0] for i in range(N)}

# C now contains the indices of data points in each cluster


In [14]:
# Print the actual data points in the first cluster (cluster 0)
cluster_0_data_points = F[C[0], :]
print(cluster_0_data_points)


[[-0.11771786  0.06197775  0.16461232 -0.01014409  0.05992496 -0.01393929
  -0.03534282 -0.04769873  0.04410837  0.04106753 -0.02749099 -0.06125814
   0.03534517 -0.08448725 -0.022216    0.09871544  0.08144165  0.08025961
  -0.10571994 -0.08709236 -0.18681227 -0.01283024  0.1199941  -0.09082173
  -0.08853832 -0.04954021 -0.0236061   0.03003179 -0.21550414 -0.05746786
  -0.09487294 -0.0589145 ]
 [-0.11791299  0.06209087  0.16486782 -0.01016495  0.06005875 -0.01395469
  -0.03539805 -0.04778659  0.0441496   0.04109329 -0.02752722 -0.06135073
   0.03540365 -0.08464003 -0.02223301  0.098897    0.08159404  0.08039114
  -0.10587427 -0.08721318 -0.18718076 -0.01282084  0.12019561 -0.09104094
  -0.08869704 -0.04964301 -0.02360645  0.03006684 -0.21591537 -0.05757183
  -0.09506587 -0.05901379]
 [-0.11803263  0.0621652   0.16504881 -0.01017828  0.06013687 -0.01397592
  -0.03542508 -0.04783311  0.04417299  0.04109842 -0.02754165 -0.0613801
   0.03544609 -0.08472417 -0.02223385  0.09904138  0.081686

In [15]:
# Print all the indices and their corresponding cluster values
for cluster_idx in range(N):
    print(f"Cluster {cluster_idx}:")
    print("Indices of data points:", C[cluster_idx])
    print("Number of data points in this cluster:", len(C[cluster_idx]))
    print()


Cluster 0:
Indices of data points: [  71  509 1042 1179 1347 1392 1877 1959 2231 2309 2514 3698 4234]
Number of data points in this cluster: 13

Cluster 1:
Indices of data points: [  51  164  559  830 1147 1290 1872 2284 2358 2410 2719 2915 3254 3264
 3310 3718 4219]
Number of data points in this cluster: 17

Cluster 2:
Indices of data points: [  68  639 1745 2269 4019]
Number of data points in this cluster: 5

Cluster 3:
Indices of data points: [ 828 2114 3028 4353]
Number of data points in this cluster: 4

Cluster 4:
Indices of data points: [ 552 1122]
Number of data points in this cluster: 2

Cluster 5:
Indices of data points: [ 647 1137 1728 2106 2427 2495 2813 3309]
Number of data points in this cluster: 8

Cluster 6:
Indices of data points: [ 427 1283 1716 2199 2732 4191 4368]
Number of data points in this cluster: 7

Cluster 7:
Indices of data points: [ 209  485  969  985 1277 1906 3750 3879]
Number of data points in this cluster: 8

Cluster 8:
Indices of data points: [ 137 1253

In [16]:
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

# Load data (assuming the data is saved in a numpy array .npy file)
try:
    F = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")  # Adjust this path if necessary
except FileNotFoundError:
    print("Error: The specified file was not found.")
    exit(1)

# Set parameters
N = 675  # Number of clusters
num_samples = F.shape[0]  # Number of data points (950)
np.random.seed(2)  # Setting the random seed for reproducibility

# Perform k-means clustering
kmeans = KMeans(n_clusters=N, init='k-means++', random_state=2)
kmeans.fit(F)

# Get cluster assignments (C_IDX) and cluster centroids (CC)
C_IDX = kmeans.labels_
CC = kmeans.cluster_centers_

# Initialize binary matrix (950 x 143) with zeros
keyframe_matrix = np.zeros((num_samples, N), dtype=int)

# Loop through each cluster and find the closest data point to the centroid
for cluster_idx in range(N):
    # Get the indices of the data points in the current cluster
    cluster_indices = np.where(C_IDX == cluster_idx)[0]
    
    # Skip empty clusters (if any)
    if len(cluster_indices) == 0:
        print(f"Cluster {cluster_idx} is empty, skipping.")
        continue
    
    # Get the feature vectors for the data points in this cluster
    cluster_features = F[cluster_indices, :]
    
    # Get the centroid of this cluster
    centroid = CC[cluster_idx]
    
    # Compute the Euclidean distance between each point in the cluster and the centroid
    distances = cdist(cluster_features, centroid.reshape(1, -1), metric='euclidean')
    
    # Get the index of the closest data point to the centroid
    closest_point_idx = cluster_indices[np.argmin(distances)]
    
    # Set the corresponding entry in the binary matrix to 1 (keyframe)
    keyframe_matrix[closest_point_idx, cluster_idx] = 1

    # Optionally, print the keyframe index for each cluster
    print(f"Cluster {cluster_idx}: Keyframe index: {closest_point_idx}")
# keyframe_matrix is now a 950 x 143 binary matrix
# You can use it for further analysis or saving to a file


Cluster 0: Keyframe index: 509
Cluster 1: Keyframe index: 1290
Cluster 2: Keyframe index: 639
Cluster 3: Keyframe index: 828
Cluster 4: Keyframe index: 552
Cluster 5: Keyframe index: 1728
Cluster 6: Keyframe index: 1716
Cluster 7: Keyframe index: 985
Cluster 8: Keyframe index: 3408
Cluster 9: Keyframe index: 1314
Cluster 10: Keyframe index: 218
Cluster 11: Keyframe index: 2538
Cluster 12: Keyframe index: 402
Cluster 13: Keyframe index: 3851
Cluster 14: Keyframe index: 2353
Cluster 15: Keyframe index: 680
Cluster 16: Keyframe index: 1384
Cluster 17: Keyframe index: 2341
Cluster 18: Keyframe index: 859
Cluster 19: Keyframe index: 1785
Cluster 20: Keyframe index: 2839
Cluster 21: Keyframe index: 3629
Cluster 22: Keyframe index: 2406
Cluster 23: Keyframe index: 838
Cluster 24: Keyframe index: 4075
Cluster 25: Keyframe index: 298
Cluster 26: Keyframe index: 2608
Cluster 27: Keyframe index: 4159
Cluster 28: Keyframe index: 3772
Cluster 29: Keyframe index: 1128
Cluster 30: Keyframe index: 341

In [17]:
# Save the keyframe binary matrix
np.save(r'C:\SumMe\Augmentation\AugA\keyframe1.npy', keyframe_matrix)

In [18]:
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from scipy.io import savemat

# Load data (assuming the data is saved in a numpy array .npy file)
try:
    F = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")  # Adjust this path if necessary
except FileNotFoundError:
    print("Error: The specified file was not found.")
    exit(1)

# Check shape of input data
print("Shape of input data F:", F.shape)

# Set parameters
N = 675  # Number of clusters
num_samples = F.shape[0]  # Number of data points (should be 950)
np.random.seed(2)  # Setting the random seed for reproducibility

# Perform k-means clustering
kmeans = KMeans(n_clusters=N, init='k-means++', n_init=50, random_state=2)
kmeans.fit(F)

# Get cluster assignments (C_IDX) and cluster centroids (CC)
C_IDX = kmeans.labels_
CC = kmeans.cluster_centers_

# Check cluster assignments
unique, counts = np.unique(C_IDX, return_counts=True)
cluster_sizes = dict(zip(unique, counts))
print("Cluster sizes:", cluster_sizes)

# Initialize binary matrix (950 x 143) with zeros
keyframe_matrix = np.zeros((num_samples, N), dtype=int)

# Loop through each cluster and find the closest data point to the centroid
for cluster_idx in range(N):
    # Get the indices of the data points in the current cluster
    cluster_indices = np.where(C_IDX == cluster_idx)[0]
    
    # Skip empty clusters (if any)
    if len(cluster_indices) == 0:
        print(f"Warning: Cluster {cluster_idx} is empty, skipping.")
        continue
    
    # Get the feature vectors for the data points in this cluster
    cluster_features = F[cluster_indices, :]
    
    # Get the centroid of this cluster
    centroid = CC[cluster_idx]
    
    # Compute the Euclidean distance between each point in the cluster and the centroid
    distances = cdist(cluster_features, centroid.reshape(1, -1), metric='euclidean')
    
    # Get the index of the closest data point to the centroid
    closest_point_idx = cluster_indices[np.argmin(distances)]
    
    # Set the corresponding entry in the binary matrix to 1 (keyframe)
    keyframe_matrix[closest_point_idx, cluster_idx] = 1

# Check the shape of the keyframe matrix
print("Shape of keyframe matrix:", keyframe_matrix.shape)

# Save the matrix as a .mat file
savemat(r'C:\SumMe\Augmentation\AugA\keyframe_matrix.mat', {'keyframe_matrix': keyframe_matrix})


Shape of input data F: (4494, 32)
Cluster sizes: {0: 6, 1: 5, 2: 4, 3: 3, 4: 13, 5: 10, 6: 9, 7: 3, 8: 8, 9: 5, 10: 4, 11: 4, 12: 10, 13: 8, 14: 9, 15: 6, 16: 10, 17: 6, 18: 12, 19: 3, 20: 6, 21: 6, 22: 10, 23: 2, 24: 15, 25: 5, 26: 7, 27: 5, 28: 3, 29: 5, 30: 7, 31: 5, 32: 7, 33: 6, 34: 19, 35: 3, 36: 4, 37: 5, 38: 7, 39: 2, 40: 6, 41: 1, 42: 3, 43: 5, 44: 4, 45: 12, 46: 13, 47: 13, 48: 4, 49: 5, 50: 10, 51: 3, 52: 7, 53: 6, 54: 6, 55: 14, 56: 4, 57: 2, 58: 4, 59: 8, 60: 5, 61: 6, 62: 3, 63: 6, 64: 11, 65: 4, 66: 8, 67: 10, 68: 5, 69: 2, 70: 4, 71: 9, 72: 13, 73: 3, 74: 11, 75: 8, 76: 6, 77: 5, 78: 9, 79: 4, 80: 9, 81: 7, 82: 2, 83: 3, 84: 9, 85: 5, 86: 5, 87: 15, 88: 8, 89: 6, 90: 6, 91: 7, 92: 16, 93: 1, 94: 9, 95: 4, 96: 6, 97: 4, 98: 7, 99: 3, 100: 4, 101: 6, 102: 6, 103: 7, 104: 8, 105: 4, 106: 12, 107: 12, 108: 3, 109: 8, 110: 4, 111: 6, 112: 10, 113: 10, 114: 1, 115: 8, 116: 5, 117: 5, 118: 3, 119: 6, 120: 3, 121: 9, 122: 9, 123: 10, 124: 5, 125: 9, 126: 2, 127: 6, 128: 16, 129

In [19]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Step 1: Load the feature matrix from the .npy file
features = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")

# Step 2: (Optional) Preprocess the data if needed
# For example, you might want to scale the features to ensure the clustering works well
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Step 3: Perform Agglomerative Clustering
# You can specify the number of clusters or other hyperparameters here
n_clusters = 675  # Example, set the desired number of clusters
clustering = AgglomerativeClustering(n_clusters=n_clusters)

# Fit the model to the feature matrix
clustering_labels = clustering.fit_predict(features_scaled)

# Step 4: (Optional) Visualize the results if the features are 2D (or use dimensionality reduction)
if features_scaled.shape[1] == 2:  # If your features are 2D, you can visualize them
    plt.scatter(features_scaled[:, 0], features_scaled[:, 1], c=clustering_labels, cmap='viridis')
    plt.title("Agglomerative Clustering Results")
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

# If you have higher-dimensional features, consider using PCA to reduce to 2D for visualization
# For example:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# features_2d = pca.fit_transform(features_scaled)
# plt.scatter(features_2d[:, 0], features_2d[:, 1], c=clustering_labels, cmap='viridis')
# plt.show()

# Step 5: View the labels assigned to each data point
print("Cluster Labels:", clustering_labels)

Cluster Labels: [190 144 505 ...  24 550 472]


In [21]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

# Load the feature matrix from the .npy file
features = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")

# Optional: Preprocess the data by scaling
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Perform Agglomerative Clustering
n_clusters = 675 # Set the desired number of clusters
clustering = AgglomerativeClustering(n_clusters=n_clusters)

# Fit the model and predict the labels
clustering_labels = clustering.fit_predict(features_scaled)

# Group the indices by cluster labels
cluster_dict = {}

for idx, label in enumerate(clustering_labels):
    if label not in cluster_dict:
        cluster_dict[label] = []
    cluster_dict[label].append(idx)

# Display the clusters in the desired format
for cluster_label, indices in cluster_dict.items():
    print(f"Cluster {cluster_label}:")
    print(f"Indices of data points: {indices}")
    print(f"Number of data points in this cluster: {len(indices)}\n")

Cluster 190:
Indices of data points: [0, 1295, 2164, 2254, 2625, 3110, 3136, 4117]
Number of data points in this cluster: 8

Cluster 144:
Indices of data points: [1, 303, 1108, 2243, 2246, 2336, 2355, 2527, 2829, 3720, 4045, 4303, 4434]
Number of data points in this cluster: 13

Cluster 505:
Indices of data points: [2, 184, 743, 899, 1836]
Number of data points in this cluster: 5

Cluster 373:
Indices of data points: [3, 6, 208, 1394, 3504]
Number of data points in this cluster: 5

Cluster 597:
Indices of data points: [4, 571, 1523]
Number of data points in this cluster: 3

Cluster 179:
Indices of data points: [5, 108, 279, 539, 1941, 3035, 3851]
Number of data points in this cluster: 7

Cluster 526:
Indices of data points: [7, 72, 593, 4429]
Number of data points in this cluster: 4

Cluster 310:
Indices of data points: [8, 371, 2742, 3329, 3610, 3925, 4208, 4411]
Number of data points in this cluster: 8

Cluster 85:
Indices of data points: [9, 280, 584, 657, 895, 1222, 1259, 2337, 335

In [22]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Step 1: Load the feature matrix from the .npy file
features = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")

# Step 2: (Optional) Preprocess the data if needed
# For example, you might want to scale the features to ensure the clustering works well
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Step 3: Perform Agglomerative Clustering
# You can specify the number of clusters or other hyperparameters here
n_clusters = 675  # Example, set the desired number of clusters
clustering = AgglomerativeClustering(n_clusters=n_clusters)

# Fit the model to the feature matrix
clustering_labels = clustering.fit_predict(features_scaled)

# Step 4: (Optional) Visualize the results if the features are 2D (or use dimensionality reduction)
if features_scaled.shape[1] == 2:  # If your features are 2D, you can visualize them
    plt.scatter(features_scaled[:, 0], features_scaled[:, 1], c=clustering_labels, cmap='viridis')
    plt.title("Agglomerative Clustering Results")
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

# If you have higher-dimensional features, consider using PCA to reduce to 2D for visualization
# For example:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# features_2d = pca.fit_transform(features_scaled)
# plt.scatter(features_2d[:, 0], features_2d[:, 1], c=clustering_labels, cmap='viridis')
# plt.show()

# Step 5: View the labels assigned to each data point
print("Cluster Labels:", clustering_labels)

Cluster Labels: [190 144 505 ...  24 550 472]


In [23]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

# Load the feature matrix from the .npy file
features = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")

# Optional: Preprocess the data by scaling
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Perform Agglomerative Clustering
n_clusters =675  # Set the desired number of clusters
clustering = AgglomerativeClustering(n_clusters=n_clusters)

# Fit the model and predict the labels
clustering_labels = clustering.fit_predict(features_scaled)

# Group the indices by cluster labels
cluster_dict = {}

for idx, label in enumerate(clustering_labels):
    if label not in cluster_dict:
        cluster_dict[label] = []
    cluster_dict[label].append(idx)

# Display the clusters in the desired format
for cluster_label, indices in cluster_dict.items():
    print(f"Cluster {cluster_label}:")
    print(f"Indices of data points: {indices}")
    print(f"Number of data points in this cluster: {len(indices)}\n")

Cluster 190:
Indices of data points: [0, 1295, 2164, 2254, 2625, 3110, 3136, 4117]
Number of data points in this cluster: 8

Cluster 144:
Indices of data points: [1, 303, 1108, 2243, 2246, 2336, 2355, 2527, 2829, 3720, 4045, 4303, 4434]
Number of data points in this cluster: 13

Cluster 505:
Indices of data points: [2, 184, 743, 899, 1836]
Number of data points in this cluster: 5

Cluster 373:
Indices of data points: [3, 6, 208, 1394, 3504]
Number of data points in this cluster: 5

Cluster 597:
Indices of data points: [4, 571, 1523]
Number of data points in this cluster: 3

Cluster 179:
Indices of data points: [5, 108, 279, 539, 1941, 3035, 3851]
Number of data points in this cluster: 7

Cluster 526:
Indices of data points: [7, 72, 593, 4429]
Number of data points in this cluster: 4

Cluster 310:
Indices of data points: [8, 371, 2742, 3329, 3610, 3925, 4208, 4411]
Number of data points in this cluster: 8

Cluster 85:
Indices of data points: [9, 280, 584, 657, 895, 1222, 1259, 2337, 335

In [24]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

# Load the feature matrix from the .npy file
features = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")

# Optional: Preprocess the data by scaling
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Perform Agglomerative Clustering
n_clusters = 675  # Set the desired number of clusters
clustering = AgglomerativeClustering(n_clusters=n_clusters)

# Fit the model and predict the labels
clustering_labels = clustering.fit_predict(features_scaled)

# Group the indices by cluster labels
cluster_dict = {}

for idx, label in enumerate(clustering_labels):
    if label not in cluster_dict:
        cluster_dict[label] = []
    cluster_dict[label].append(idx)

# Function to get the keyframe index (centroid-based approach)
def get_keyframe_for_cluster(cluster_indices):
    # Extract the features for the current cluster
    cluster_features = features_scaled[cluster_indices]
    
    # Compute the centroid of the cluster (mean of the features)
    centroid = np.mean(cluster_features, axis=0)
    
    # Calculate the distance from each point to the centroid
    distances = cdist([centroid], cluster_features)
    
    # Get the index of the point closest to the centroid
    keyframe_index = cluster_indices[np.argmin(distances)]
    return keyframe_index

# Generate and display the sequence of keyframe indices for each cluster
keyframe_sequence = []

for cluster_label, indices in cluster_dict.items():
    keyframe_index = get_keyframe_for_cluster(indices)
    keyframe_sequence.append((cluster_label, keyframe_index))

# Print the sequence of keyframes in the desired format
for cluster_label, keyframe_index in keyframe_sequence:
    print(f"Cluster {cluster_label}: Keyframe index: {keyframe_index}")

Cluster 190: Keyframe index: 3110
Cluster 144: Keyframe index: 4303
Cluster 505: Keyframe index: 899
Cluster 373: Keyframe index: 3504
Cluster 597: Keyframe index: 4
Cluster 179: Keyframe index: 3851
Cluster 526: Keyframe index: 7
Cluster 310: Keyframe index: 371
Cluster 85: Keyframe index: 584
Cluster 233: Keyframe index: 4320
Cluster 535: Keyframe index: 578
Cluster 521: Keyframe index: 1516
Cluster 545: Keyframe index: 1762
Cluster 0: Keyframe index: 468
Cluster 461: Keyframe index: 2275
Cluster 419: Keyframe index: 3825
Cluster 317: Keyframe index: 17
Cluster 199: Keyframe index: 4071
Cluster 47: Keyframe index: 1410
Cluster 242: Keyframe index: 20
Cluster 592: Keyframe index: 3521
Cluster 639: Keyframe index: 22
Cluster 322: Keyframe index: 23
Cluster 261: Keyframe index: 1301
Cluster 274: Keyframe index: 2749
Cluster 259: Keyframe index: 1739
Cluster 250: Keyframe index: 2475
Cluster 171: Keyframe index: 1051
Cluster 46: Keyframe index: 330
Cluster 72: Keyframe index: 1086
Cluste

In [25]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
from scipy.io import savemat  # Import savemat to save in .mat format
import pandas as pd  # Import pandas for CSV file export

# Load the LSTM features from the .npy file
lstm_features = np.load(r"C:\SumMe\Augmentation\AugA\features_last_lstm.npy")

# Flatten LSTM features for clustering (if necessary)
n_samples = lstm_features.shape[0]  # Define the number of samples
n_clusters = 675  # Set the number of clusters
lstm_features_flattened = lstm_features.reshape(n_samples, -1)

# Standardizing the data before clustering
scaler = StandardScaler()
lstm_features_scaled = scaler.fit_transform(lstm_features_flattened)

# Apply Agglomerative Clustering with 'ward' linkage (no need to specify affinity)
agg_clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = agg_clustering.fit_predict(lstm_features_scaled)

# Group data points by cluster
clustered_data = {}

for idx, label in enumerate(labels):
    if label not in clustered_data:
        clustered_data[label] = []
    clustered_data[label].append(idx)

# Find keyframe for each cluster
keyframes = {}

for cluster_id, indices in clustered_data.items():
    # Get the feature vectors for the data points in this cluster
    cluster_features = lstm_features_scaled[indices]
    
    # Compute the centroid of the cluster (average of all feature vectors in the cluster)
    centroid = np.mean(cluster_features, axis=0)
    
    # Compute the Euclidean distance from each data point to the centroid
    distances = cdist(cluster_features, [centroid], metric='euclidean')
    
    # Find the index of the closest data point (keyframe)
    keyframe_idx = indices[np.argmin(distances)]
    
    # Store the keyframe index for the cluster
    keyframes[cluster_id] = keyframe_idx

# Create a binary matrix of shape (n_samples, n_clusters)
keyframe_matrix = np.zeros((n_samples, n_clusters), dtype=int)

# Mark the keyframes (set the corresponding index to 1)
for cluster_id, keyframe_idx in keyframes.items():
    keyframe_matrix[keyframe_idx, cluster_id] = 1

# Save the keyframe binary matrix as a .mat file
keyframe_data = {'keyframe_matrix': keyframe_matrix}  # Create a dictionary to save in .mat format
savemat(r'C:\SumMe\Augmentation\AugA\features_last_lstm1.mat', keyframe_data)

# Print confirmation for .mat file
print(f"Keyframe binary matrix saved to C:/SumMe/Spectral/AugJ/features_last_lstm1.mat")
print(f"Dimensions of the keyframe binary matrix: {keyframe_matrix.shape}")

# Save the keyframe binary matrix as a .csv file
keyframe_matrix_df = pd.DataFrame(keyframe_matrix)  # Convert matrix to DataFrame
keyframe_matrix_df.to_csv(r'C:\SumMe\Augmentation\AugA\features_last_lstm1.csv', index=False)

# Print confirmation for .csv file
print(f"Keyframe binary matrix saved to C:/SumMe/Spectral/AugJ/features_last_lstm1.csv")

# Print the sequence of keyframe indices in order of cluster numbers (Cluster 0, Cluster 1, ..., Cluster n)
print("\nKeyframe indices for each cluster:")
for cluster_id in range(n_clusters):
    if cluster_id in keyframes:
        print(f"Cluster {cluster_id}: Keyframe Index {keyframes[cluster_id]}")
    else:
        print(f"Cluster {cluster_id}: No keyframe found")

Keyframe binary matrix saved to C:/SumMe/Spectral/AugJ/features_last_lstm1.mat
Dimensions of the keyframe binary matrix: (4494, 675)
Keyframe binary matrix saved to C:/SumMe/Spectral/AugJ/features_last_lstm1.csv

Keyframe indices for each cluster:
Cluster 0: Keyframe Index 468
Cluster 1: Keyframe Index 1092
Cluster 2: Keyframe Index 1072
Cluster 3: Keyframe Index 888
Cluster 4: Keyframe Index 1695
Cluster 5: Keyframe Index 1386
Cluster 6: Keyframe Index 2989
Cluster 7: Keyframe Index 1556
Cluster 8: Keyframe Index 1389
Cluster 9: Keyframe Index 2237
Cluster 10: Keyframe Index 3180
Cluster 11: Keyframe Index 1378
Cluster 12: Keyframe Index 1440
Cluster 13: Keyframe Index 829
Cluster 14: Keyframe Index 3962
Cluster 15: Keyframe Index 211
Cluster 16: Keyframe Index 3246
Cluster 17: Keyframe Index 439
Cluster 18: Keyframe Index 4058
Cluster 19: Keyframe Index 270
Cluster 20: Keyframe Index 3842
Cluster 21: Keyframe Index 2046
Cluster 22: Keyframe Index 2509
Cluster 23: Keyframe Index 275
C

In [26]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import scipy.io

# Step 1: Load the annotator labels from the .mat file
annotator_path = r"C:\SumMe\GT\Air_Force_One.mat" # Path to the .mat file
mat_data = scipy.io.loadmat(annotator_path)

# Check the keys in the .mat file
print("Keys in the .mat file:", mat_data.keys())


user_scores = mat_data['user_score']  
print(f"Shape of user_score: {user_scores.shape}")

# Step 2: Load our model's predictions from the .csv file
our_path = r"C:\SumMe\Augmentation\AugA\features_last_lstm1.csv" # Path to the CSV file
u1 = pd.read_csv(our_path, header=None).values.flatten()  # Our model's predictions (binary array)


print(f"Shape of model's predictions (u1): {u1.shape}")

# Ensure that the length of u1 matches the number of rows in user_scores
if u1.shape[0] != user_scores.shape[0]:
    print(f"Warning: Mismatched lengths. Adjusting the data...")
    u1 = u1[:user_scores.shape[0]]  # Truncate u1 to match the number of rows in user_scores

# Step 3: Ensure both u1 and user_scores are binary (logical values, 0 or 1)
u1 = u1.astype(bool)

# Step 4: Compute the F1-Score for each of the 15 datasets (columns in user_scores)
f1_scores = []  # List to store F1 scores for each dataset

for i in range(user_scores.shape[1]):  # Loop through the 15 columns
    u2 = user_scores[:, i]  # Extract the i-th column (ground truth labels for the i-th dataset)
    
    # Ensure u2 is a boolean array
    u2 = u2.astype(bool)
    
    # Step 5: Compute the confusion matrix
    cm = confusion_matrix(u2, u1)
    
    # Extract the confusion matrix components
    tn, fp, fn, tp = cm.ravel()  # Unpack the confusion matrix
    
    # Debugging step: Print confusion matrix for the 4th column
    if i == 3:  # Checking the 4th column
        print(f"Confusion Matrix for user_score column {i+1} (4th column):")
        print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
    
    # Check for zero denominators and calculate Precision, Recall, and F1-Score safely
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # F1 Score is calculated if both precision and recall are non-zero, otherwise it's 0 or nan
    if precision + recall > 0:
        fscore = 2 * (precision * recall) / (precision + recall)
    else:
        fscore = 0.0  # F1 score is 0 if either precision or recall is zero
    
    f1_scores.append(fscore)  # Append F1 score for this dataset
    
    # Print the F1 Score for this dataset
    print(f"\nF1-Score for user_score column {i+1}: {fscore:.4f}")

# Step 6: Output all the F1 scores for the 15 datasets
print("\nAll F1 Scores:")
for i, score in enumerate(f1_scores, 1):
    print(f"F1 Score for user_score column {i}: {score:.4f}")

Keys in the .mat file: dict_keys(['__header__', '__version__', '__globals__', 'all_userIDs', 'segments', 'nFrames', 'video_duration', 'FPS', 'gt_score', 'user_score'])
Shape of user_score: (4494, 15)
Shape of model's predictions (u1): (3034125,)

F1-Score for user_score column 1: 0.1507

F1-Score for user_score column 2: 0.1011

F1-Score for user_score column 3: 0.0385
Confusion Matrix for user_score column 4 (4th column):
TN: 3161, FP: 661, FN: 658, TP: 14

F1-Score for user_score column 4: 0.0208

F1-Score for user_score column 5: 0.2076

F1-Score for user_score column 6: 0.1941

F1-Score for user_score column 7: 0.0000

F1-Score for user_score column 8: 0.0000

F1-Score for user_score column 9: 0.0015

F1-Score for user_score column 10: 0.0716

F1-Score for user_score column 11: 0.1453

F1-Score for user_score column 12: 0.0000

F1-Score for user_score column 13: 0.0000

F1-Score for user_score column 14: 0.0512

F1-Score for user_score column 15: 0.0016

All F1 Scores:
F1 Score for